# ARMT with Llama-3.2-1B!

## Create a model from config in ARMT official repo

In [3]:
# Minimal example: wrap Llama-3.2-1B with ARMT and run a quick forward pass
import os
import torch
from transformers import AutoTokenizer
from modeling_amt.model import ARMTConfig, ARMTForCausalLM

# Base model to wrap
base_model_name = "meta-llama/Llama-3.2-1B"

# Minimal ARMT config, similar to run_finetuning_lm_rmt_hf_armt.py
segment_size = 128
armt_cfg = ARMTConfig(
    base_model_name=base_model_name,
    num_mem_tokens=16,
    d_mem=64,
    segment_size=segment_size,
    segment_alignment="left",
    sliding_window=True,
    attend_to_previous_input=False,
    use_sink=True,
    layers_attr="model.layers",  # Llama layers path
    wrap_pos=False,
    correction=True,
    n_heads=1,
    use_denom=True,
    gating=False,
    freeze_mem=False,
)

# Build wrapped model
model = ARMTForCausalLM(armt_cfg)
model.eval()

# Tokenizer and a tiny test batch
# Llama tokenizer may not have pad_token by default, set it to eos for batching
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer([
    "Hello Llama with ARMT!",
    "Testing ARMT wrapping on Llama-3.2-1B.",
], return_tensors="pt", padding=True, truncation=True, max_length=segment_size)

labels = inputs["input_ids"].clone()
with torch.no_grad():
    out = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        labels=labels,
    )
print("loss:", float(out.loss))


loss: 8.00378131866455


## Importnat: now model needs to be trained. The memory weights are initialized from scratch.

## Pushing to hub with the corresponding code

In [4]:
# Push the wrapped ARMT+Llama model to Hugging Face Hub (single-file modeling module)
import os
import json
import shutil
from pathlib import Path
from tempfile import TemporaryDirectory

from huggingface_hub import HfApi, upload_folder
from transformers import AutoModelForCausalLM

# Change to your namespace/repo name if desired
repo_id = "irodkin/armt-sw-llama-3.2-1b-untrained_single"

# Local source for ARMT code in this repo (relative to this notebook directory)
local_modeling_dir = Path("./modeling_amt")
assert (local_modeling_dir / "model.py").exists(), "Expected modeling_amt/model.py next to this notebook"

with TemporaryDirectory() as tmpdir:
    repo_dir = Path(tmpdir) / "repo"
    repo_dir.mkdir(parents=True, exist_ok=True)

    # Save model weights and config
    model.save_pretrained(repo_dir)

    # Inline ARMT code into one file to avoid external imports
    act_utils_path = local_modeling_dir / "act_utils.py"
    lm_path       = local_modeling_dir / "language_modeling.py"
    model_path    = local_modeling_dir / "model.py"

    act_code = act_utils_path.read_text()
    lm_code  = lm_path.read_text()
    model_code = model_path.read_text()

    # Make inlined code self-contained (remove intra-package imports)
    lm_code = lm_code.replace("from modeling_amt.act_utils import", "# inlined act_utils: removed import")
    model_code = model_code.replace("from modeling_amt.language_modeling import", "# inlined language_modeling: removed import")

    single_file = repo_dir / "modeling_armt.py"
    single_file.write_text(
        "# === Inlined ARMT for HF Hub (single-file) ===\n\n"
        + "# ---- act_utils.py ----\n" + act_code + "\n\n"
        + "# ---- language_modeling.py ----\n" + lm_code + "\n\n"
        + "# ---- model.py ----\n" + model_code + "\n"
    )

    # Ensure auto_map points to the single-file module
    cfg_path = repo_dir / "config.json"
    cfg = json.loads(cfg_path.read_text())
    cfg["architectures"] = ["ARMTForCausalLM"]
    cfg["auto_map"] = {
        "AutoConfig": "modeling_armt.ARMTConfig",
        "AutoModelForCausalLM": "modeling_armt.ARMTForCausalLM",
    }
    cfg_path.write_text(json.dumps(cfg, indent=2))

    # README
    (repo_dir / "README.md").write_text(
        f"# {repo_id}\n\nARMT-wrapped Llama-3.2-1B demo (single-file). Load with trust_remote_code=True. WARNING: This model is initialized with random weights."
    )

    # Create or update repo and upload
    api = HfApi()
    api.create_repo(repo_id, private=True, exist_ok=True)
    upload_folder(
        folder_path=str(repo_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload ARMT+Llama-3.2-1B demo (single-file)",
    )

print(f"Pushed to {repo_id}")



/home/ivan.rodkin/miniconda3/envs/env/lib/python3.9/site-packages/huggingface_hub/hf_api.py:9696: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


model-00002-of-00002.safetensors:   0%|          | 0.00/243M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Pushed to irodkin/armt-sw-llama-3.2-1b-untrained_single


## Now you can load the model anywhere

In [5]:
from transformers import AutoModelForCausalLM

loaded_model = AutoModelForCausalLM.from_pretrained(
    "irodkin/armt-sw-llama-3.2-1b-untrained_single", trust_remote_code=True
)

loaded_model.tie_weights()

config.json:   0%|          | 0.00/839 [00:00<?, ?B/s]

modeling_armt.py:   0%|          | 0.00/73.3k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/irodkin/armt-sw-llama-3.2-1b-untrained_single:
- modeling_armt.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


*** Can't import RWKV model ***


model.safetensors.index.json:   0%|          | 0.00/23.8k [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/243M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of ARMTForCausalLM were not initialized from the model checkpoint at irodkin/armt-sw-llama-3.2-1b-untrained_single and are newly initialized: ['armt.memory_cell.model.lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# INNER LOOP ARMT

In [ ]:
from modeling_amt.inner_loop import InnerLoopARMTForCausalLM
from modeling_amt.inner_loop import ARMTConfig
import torch 

# model_path = "../runs/HuggingFaceFW/fineweb-edu/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_8x1024_mem32_bs64_hf_armt_dmem64/run_20/checkpoint-2500"
model_path = "irodkin/run_21"

config = ARMTConfig.from_pretrained(model_path)

# model = InnerLoopARMTForCausalLM.from_pretrained(
#     model_path,
# )

model = InnerLoopARMTForCausalLM(config)

config.wrap_layers = [1,] * 16

model


InnerLoopARMTForCausalLM(
  (model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(128256, 2048)
      (layers): ModuleList(
        (0-15): 16 x InnerLoopAssociativeLayerWrapper(
          (layer): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
              (k_proj): Linear(in_features=2048, out_features=512, bias=False)
              (v_proj): Linear(in_features=2048, out_features=512, bias=False)
              (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            )
            (mlp): LlamaMLP(
              (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
              (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
              (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
              (act_fn): SiLU()
            )
            (input_layernorm): LlamaRMSNorm((2048,), eps=1e

In [11]:
# Push the wrapped ARMT+Llama model to Hugging Face Hub (single-file modeling module)
import os
import json
import shutil
from pathlib import Path
from tempfile import TemporaryDirectory

from huggingface_hub import HfApi, upload_folder
from transformers import AutoModelForCausalLM

# Change to your namespace/repo name if desired
repo_id = "irodkin/test"

# Local source for ARMT code in this repo (relative to this notebook directory)
local_modeling_dir = Path("./modeling_amt")
assert (local_modeling_dir / "model.py").exists(), "Expected modeling_amt/model.py next to this notebook"

with TemporaryDirectory() as tmpdir:
    repo_dir = Path(tmpdir) / "repo"
    repo_dir.mkdir(parents=True, exist_ok=True)

    # Save model weights and config
    model.save_pretrained(repo_dir)

    # Inline ARMT code into one file to avoid external imports
    inner_loop_path = local_modeling_dir / "inner_loop.py"

    inner_loop_code = inner_loop_path.read_text()
    # Make inlined code self-contained (remove intra-package imports)
    inner_loop_code = inner_loop_code.replace("from modeling_amt.model import ARMTConfig", "# inlined model: removed import")

    single_file = repo_dir / "modeling_armt.py"
    single_file.write_text(
        "# === Inlined ARMT for HF Hub (single-file) ===\n\n"
        + "# ---- inner_loop.py ----\n" + inner_loop_code + "\n"
    )

    # Ensure auto_map points to the single-file module
    cfg_path = repo_dir / "config.json"
    cfg = json.loads(cfg_path.read_text())
    cfg["architectures"] = ["InnerLoopARMTForCausalLM"]
    cfg["auto_map"] = {
        "AutoConfig": "modeling_armt.ARMTConfig",
        "AutoModelForCausalLM": "modeling_armt.InnerLoopARMTForCausalLM",
    }
    cfg_path.write_text(json.dumps(cfg, indent=2))

    # README
    (repo_dir / "README.md").write_text(
        f"# {repo_id}\n\nARMT-wrapped Llama-3.2-1B. Trained on FineWeb-edu with 1B tokens. 8 segments, 1024 tokens each."
    )

    # Create or update repo and upload
    api = HfApi()
    api.create_repo(repo_id, private=True, exist_ok=True)
    upload_folder(
        folder_path=str(repo_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload ARMT+Llama-3.2-1B",
    )

print(f"Pushed to {repo_id}")

/home/ivan.rodkin/miniconda3/envs/env/lib/python3.9/site-packages/huggingface_hub/hf_api.py:9696: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...po/model-00002-of-00002.safetensors:   1%|1         | 3.52MB /  236MB            

  ...po/model-00001-of-00002.safetensors:   1%|          | 41.9MB / 4.99GB            

Pushed to irodkin/test


In [12]:
from transformers import AutoModelForCausalLM, AutoConfig

config = AutoConfig.from_pretrained(
    "irodkin/test", trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    "irodkin/test", trust_remote_code=True
)

model

config.json:   0%|          | 0.00/987 [00:00<?, ?B/s]

modeling_armt.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/irodkin/test:
- modeling_armt.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


*** Can't import liger_kernel ***


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/236M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

InnerLoopARMTForCausalLM(
  (model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(128256, 2048)
      (layers): ModuleList(
        (0-15): 16 x InnerLoopAssociativeLayerWrapper(
          (layer): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
              (k_proj): Linear(in_features=2048, out_features=512, bias=False)
              (v_proj): Linear(in_features=2048, out_features=512, bias=False)
              (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            )
            (mlp): LlamaMLP(
              (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
              (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
              (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
              (act_fn): SiLU()
            )
            (input_layernorm): LlamaRMSNorm((2048,), eps=1e

In [13]:
model(torch.randint(0, 100, (1, 10)), attention_mask=torch.ones(1, 10))

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 3.2354,  6.4598,  7.2019,  ..., -2.3266, -2.3272, -2.3273],
         [ 5.8142,  8.9760,  8.8775,  ..., -1.3137, -1.3142, -1.3142],
         [ 6.8251, 11.7376,  9.4706,  ..., -0.5671, -0.5677, -0.5678],
         ...,
         [ 9.0215, 14.0139, 12.6245,  ...,  0.5407,  0.5404,  0.5402],
         [ 8.6106, 13.5007, 11.6650,  ...,  0.4150,  0.4146,  0.4143],
         [ 7.8559, 12.1166, 11.9704,  ...,  0.8383,  0.8381,  0.8376]]],
       grad_fn=<CatBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None)